In [27]:
import numpy as np

In [28]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]


In [29]:
for _ in data:
    if _[3] == 'Wine':
        _[3] = 0
    elif _[3] =='Beer':
        _[3] = 1
    elif _[3] =='Whiskey':
        _[3] = 2

y_train = np.array([_[3] for _ in data])
X_train = np.array([_[0:3] for _ in data], dtype=np.float32)


In [30]:
print(X_train, y_train)

[[12.   1.5  1. ]
 [ 5.   2.   0. ]
 [40.   0.   1. ]
 [13.5  1.2  1. ]
 [ 4.5  1.8  0. ]
 [38.   0.1  1. ]
 [11.5  1.7  1. ]
 [ 5.5  2.3  0. ]] [0 1 2 0 1 2 0 1]


In [31]:
#function to calculate Gini impurity
# gini  = 1 - sum(pi**2)
def gini(y):
    y = np.array(y)
    n = len(y)
    if n==0:
        return 0.0

    classes, cnt = np.unique(y, return_counts=True)
    n =cnt.sum()
    p = cnt/n
    ans = 1 - np.sum((p**2))
    return ans



In [32]:
def entropy(y):
    y = np.array(y)
    n = len(y)
    if n==0:
        return 0.0
    classes, cnt = np.unique(y, return_counts=True)
    p = cnt/n
    ans = -np.sum(p*np.log2(p))
    return ans
# entropy also usefully for uncertainity of split
# uncertainity = 0 -> more then sufficient knowledge

In [33]:
# Split Finder

def split_finder(x, y, func='gini'):
    x = np.array(x)
    y = np.array(y)

    n_sample, n_feature = x.shape
    if func == 'gini':
        split_func = gini
    elif func == 'entropy':
        split_func = entropy

    best_feature = -1
    best_score = np.inf
    best_threshold = None

    for feature in range(n_feature):
        val = np.unique(x[:, feature])
        for threshold in val:
            l = x[:, feature] <= threshold
            r = x[:, feature] > threshold

            y_left = y[l]
            y_right = y[r]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            f_left = split_func(y_left)
            f_right = split_func(y_right)

            w_left = len(y_left)/n_sample
            w_right = len(y_right)/n_sample
            score = w_left*f_left + w_right*f_right

            if score < best_score:
                best_score = score
                best_feature = feature
                best_threshold = threshold
    if best_feature == -1:
        return None, None, None

    return best_feature, best_threshold, best_score



In [34]:

def train_test_split(X, y, random_state=None):
    X = np.array(X)
    y = np.array(y)
    if random_state is not None:
        np.random.seed(random_state)

    # suffling
    idx = np.arange(len(X))
    np.random.shuffle(idx)
    test_size = 0.3

    split_idx = int(len(X)*(1-test_size))
    train_idx = idx[:split_idx]
    test_idx = idx[split_idx:]

    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    return X_train, y_train, X_test, y_test



In [35]:
def accuracy(y_true, y_pred):
    x = np.sum(y_true == y_pred)
    return (x/len(y_true))*100


In [36]:
def z_score_normalisation(x):
    X = np.array(x, dtype=float)
    std = np.std(X, axis=0)
    std[std==0] = 1

    X_norm = (X - np.mean(X, axis=0)) / std
    return X_norm

In [37]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None, info_gain=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.info_gain = info_gain



In [38]:
class DecisionTree:
    def __init__(self, max_depth=10, min_samples=1, func='gini'):
        self.root = None
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.func = func

    def majority(self, y):
        cnt = np.bincount(np.array(y))
        ans = int(np.argmax(cnt))
        return ans

    def build(self, X, y, depth):
        p = len(y)
        q = len(set(y))
        if p <= self.min_samples or depth >= self.max_depth or q == 1:
            return Node(value=self.majority(y))

        feature, threshold, score = split_finder(X, y, self.func)

        if feature is None or threshold is None:
            return Node(value=self.majority(y))

        left = X[:,feature] <= threshold
        right = ~left
        left_child = self.build(X[left],y[left],depth+1)
        right_child = self.build(X[right],y[right],depth+1)

        return Node(feature=feature, threshold=threshold, left=left_child, right=right_child,info_gain=score)

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self.root = self.build(X,y,0)

    def traverse(self, node, x):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self.traverse(node.left, x)
        else:
            return self.traverse(node.right, x)


    def predict_one(self, x):
        return self.traverse(self.root, x)

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

In [39]:
def minmax(x):
    x = np.array(x)
    min,max = x.min(axis=0),x.max(axis=0)
    return (x-min)/(max-min+1e-8)

In [40]:
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])


In [41]:
zscore_normalised_test = z_score_normalisation(test_data)
zscore_normalised_x_train = z_score_normalisation(X_train)

minmax_test_data_normalised = minmax(test_data)
minmax_x_train_normalised = minmax(X_train)


In [42]:
model = DecisionTree(max_depth=15, func='gini')
model.fit(zscore_normalised_x_train, y_train)
y_pred = model.predict(zscore_normalised_test)
# y_pred = model.predict(test_data_normalised)

print("Predictions| with z-score normalised data| :", y_pred)
labels = ['Wine', 'Beer', 'Whiskey']
print([labels[i] for i in y_pred])

Predictions| with z-score normalised data| : [1 2 0]
['Beer', 'Whiskey', 'Wine']


In [45]:
model = DecisionTree(max_depth=15, func='entropy')
model.fit(zscore_normalised_x_train, y_train)
y_pred = model.predict(zscore_normalised_test)
# y_pred = model.predict(test_data_normalised)

print("Predictions| with (Entropy function) z-score normalised data| :", y_pred)
labels = ['Wine', 'Beer', 'Whiskey']
print([labels[i] for i in y_pred])

Predictions| with (Entropy function) z-score normalised data| : [1 2 0]
['Beer', 'Whiskey', 'Wine']


In [43]:
model1 = DecisionTree(max_depth=20, func='gini')
model1.fit(minmax_x_train_normalised, y_train)
y_pred1 = model1.predict(minmax_test_data_normalised)
print("Predictions | with minmax noralised data | ", y_pred1)
labels = ['Wine', 'Beer', 'Whiskey']
print([labels[i] for i in y_pred1])

Predictions | with minmax noralised data |  [1 2 0]
['Beer', 'Whiskey', 'Wine']


In [46]:
model1 = DecisionTree(max_depth=20, func='entropy')
model1.fit(minmax_x_train_normalised, y_train)
y_pred1 = model1.predict(minmax_test_data_normalised)
print("Predictions | with(Entropy functin) minmax noralised data | ", y_pred1)
labels = ['Wine', 'Beer', 'Whiskey']
print([labels[i] for i in y_pred1])

Predictions | with(Entropy functin) minmax noralised data |  [1 2 0]
['Beer', 'Whiskey', 'Wine']


In [47]:
model2 = DecisionTree(max_depth=20, func='entropy')
model2.fit(X_train, y_train)
y_pred2 = model2.predict(test_data)
print("With (entropy) function and normal data|", y_pred2)
labels = ['Wine', 'Beer', 'Whiskey']
print([labels[i] for i in y_pred2])

With (entropy) function and normal data| [0 2 0]
['Wine', 'Whiskey', 'Wine']


In [48]:
model2 = DecisionTree(max_depth=20, func='gini')
model2.fit(X_train, y_train)
y_pred2 = model2.predict(test_data)
print("With (gini) function and normal data|", y_pred2)
labels = ['Wine', 'Beer', 'Whiskey']
print([labels[i] for i in y_pred2])

With (gini) function and normal data| [0 2 0]
['Wine', 'Whiskey', 'Wine']


In [63]:
# tree
labels = ['Wine', 'Beer', 'Whiskey']
def beuatiful_tree(node, feature, depth =0):
    gap = "    "*depth

    if node.value is not None:
        x = labels[node.value]
        print(gap + f"Predict : {x}")
        return
    feature = feature[node.feature]
    print(gap + f"If > {feature} <= {node.threshold}")
    beuatiful_tree(node.left, feature, depth+1)
    print(gap + f"Else > {feature} > {node.threshold}")
    beuatiful_tree(node.right, feature, depth+1)

In [64]:
features = ['Alcohol', 'Sugar', 'Color']
beuatiful_tree(model1.root, features)

If > Alcohol <= 0.028169013559818268
    Predict : Beer
Else > Alcohol > 0.028169013559818268
    If > A <= 0.2535211145877838
        Predict : Wine
    Else > A > 0.2535211145877838
        Predict : Whiskey


In [65]:
# max_dept changing

for _ in range(1, 11):
    p = DecisionTree(max_depth=_, func='gini')
    p.fit(zscore_normalised_x_train, y_train)
    pred = p.predict(zscore_normalised_test)
    print(f"Depth {_} : Accurry : {accuracy([1 ,2, 0], pred)}")

# y_true = [1, 2, 0]
# # Beer whiskey wine


Depth 1 : Accurry : 66.66666666666666
Depth 2 : Accurry : 100.0
Depth 3 : Accurry : 100.0
Depth 4 : Accurry : 100.0
Depth 5 : Accurry : 100.0
Depth 6 : Accurry : 100.0
Depth 7 : Accurry : 100.0
Depth 8 : Accurry : 100.0
Depth 9 : Accurry : 100.0
Depth 10 : Accurry : 100.0
